# Diagnose alibrex setup

Walks through every check the package performs at import time, so
you can verify your install or troubleshoot if `from alibrex import …`
fails.

Safe to run **with or without** Alibre Design open — Section 3
inspects the DLL path without needing a live connection.

## 1. Environment

In [1]:
import sys, platform

print(f"Python:     {sys.version.split()[0]}")
print(f"Platform:   {platform.platform()}")
print(f"Arch:       {platform.machine()}")

Python:     3.13.11
Platform:   Windows-11-10.0.26200-SP0
Arch:       AMD64


## 2. Package versions

In [2]:
from importlib.metadata import version, PackageNotFoundError

for pkg in ("alibrex", "pythonnet"):
    try:
        print(f"  {pkg:12s} {version(pkg)}")
    except PackageNotFoundError:
        print(f"  {pkg:12s} NOT INSTALLED")

  alibrex      1.0.0
  pythonnet    3.0.5


## 3. AlibreX.dll discovery

The package tries four sources in order. The first hit wins for
actually loading the DLL, but this notebook reports **every**
source — useful for confirming your install registers cleanly.

1. `$ALIBREX_DLL` environment variable
2. Windows Registry COM ProgID (`AlibreX.AutomationHook` → CLSID)
3. Windows Registry Alibre install key (`HKLM\SOFTWARE\Alibre, Inc.\Alibre Design\<ver>\HomeDirectory`)
4. `%ProgramFiles%` glob (`Alibre Design*\Program\AlibreX.dll`)

In [3]:
import os

# Let us inspect the DLL discovery without needing Alibre to be running.
os.environ["ALIBREX_SKIP_RUNNING_CHECK"] = "1"

import alibrex
from alibrex._discover import discover_sources

sources = discover_sources()
labels = {
    "env_var":          "Source 1 ($ALIBREX_DLL):      ",
    "com_registry":     "Source 2 (Registry COM):      ",
    "alibre_registry":  "Source 3 (Alibre install reg):",
    "program_files":    "Source 4 (%ProgramFiles%):    ",
}
for key, label in labels.items():
    print(f"{label}  {sources[key] or '(no hit)'}")

resolved = alibrex._DLL_PATH
print(f"\nResolved DLL: {resolved}")

Source 1 ($ALIBREX_DLL):        C:\Program Files\Alibre Design 29.0.0.29053-BETA-2\Program\AlibreX.dll
Source 2 (Registry COM):        (no hit)
Source 3 (Alibre install reg):  C:\Program Files\Alibre Design 29.0.0.29053-BETA-2\Program\AlibreX.dll
Source 4 (%ProgramFiles%):      C:\Program Files\Alibre Design 29.0.0.29053-BETA-2\Program\AlibreX.dll

Resolved DLL: C:\Program Files\Alibre Design 29.0.0.29053-BETA-2\Program\AlibreX.dll


In [4]:
# List EVERY source whose hit matches the resolved DLL.
# Multiple sources can match the same install (e.g. registry + program-files
# both point at the same DLL); the package uses the FIRST hit but a healthy
# setup usually has more than one source agreeing.
matches = [
    labels[key].strip().rstrip(':').strip()
    for key, hit in sources.items()
    if hit and os.path.normcase(str(hit)) == os.path.normcase(resolved)
]
if matches:
    print(f"-> Sources that match the resolved DLL ({len(matches)}):")
    for m in matches:
        print(f"     - {m}")
else:
    print("-> No source matches the resolved DLL (unexpected)")

-> Sources that match the resolved DLL (3):
     - Source 1 ($ALIBREX_DLL)
     - Source 3 (Alibre install reg)
     - Source 4 (%ProgramFiles%)


## 4. Live connection

Now drop the bypass and try a real connection. If Alibre Design is
running, you'll see the version and a summary of open documents.

In [5]:
os.environ.pop("ALIBREX_SKIP_RUNNING_CHECK", None)

from alibrex import connect_to_running_alibre

root = connect_to_running_alibre()
print(f"Alibre version: {root.Version}")
print(f"Open sessions:  {root.Sessions.Count}")

Alibre version: PRODUCTVERSION 29,0,0,29053
Open sessions:  1


In [6]:
s = root.TopmostSession
if s is not None:
    print(f"Active doc:     {s.Name}  type={s.SessionType}")
else:
    print(f"Active doc:     (none open)")

Active doc:     cl1  type=AD_PART
